# 05 - LSTM Bidireccional con Atencion (PyTorch)

## Objetivo
Entrenar el modelo principal de prediccion de fallas a 24 horas.

## Arquitectura
- BiLSTM(128) x 2 capas + Attention + FC(64) -> Sigmoid
- Input: (batch, 48 timesteps, 12 features)
- Output: Probabilidad de falla en proximas 24h
- ~400K parametros

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, roc_auc_score

from src.data.preprocessor import FiberDataPreprocessor
from src.data.feature_engineer import FiberFeatureEngineer
from src.data.dataset import create_dataloaders
from src.models.lstm_predictor import LSTMPredictor
from src.utils.config_loader import load_config, get_path
from src.utils.visualization import apply_style, plot_training_history

apply_style()
config = load_config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

In [ ]:
# Preparar datos
preprocessor = FiberDataPreprocessor()
train_df, val_df, test_df = preprocessor.process_pipeline()

# Features para el LSTM (solo metricas de sensor + log_ber)
feature_cols = [
    'optical_power_dbm', 'attenuation_db_km', 'ber', 'osnr_db',
    'chromatic_dispersion', 'temperature', 'humidity', 'log_ber',
    'hour', 'day_of_week', 'is_fault', 'pre_fault_hours'
]
# Filtrar a columnas disponibles
feature_cols = [c for c in feature_cols if c in train_df.columns]
# Remover columnas target de features
feature_cols = [c for c in feature_cols if c not in ['is_fault', 'fault_in_24h', 'fault_label', 'pre_fault_hours']]

print(f'Features: {feature_cols}')
print(f'N features: {len(feature_cols)}')

In [ ]:
# Crear DataLoaders
cfg = config['models']['lstm_predictor']
seq_len = config['preprocessing']['sequence_length']

train_loader, val_loader, test_loader = create_dataloaders(
    train_df, val_df, test_df,
    feature_columns=feature_cols,
    target_column='fault_in_24h',
    sequence_length=seq_len,
    batch_size=cfg['batch_size'],
    dataset_type='sequence',
)

# Verificar shapes
batch_x, batch_y = next(iter(train_loader))
print(f'Batch X shape: {batch_x.shape}')  # (batch, seq_len, n_features)
print(f'Batch Y shape: {batch_y.shape}')  # (batch,)

In [ ]:
# Crear modelo
model = LSTMPredictor(
    n_features=len(feature_cols),
    hidden_size=cfg['hidden_size'],
    num_layers=cfg['num_layers'],
    dropout=cfg['dropout'],
    bidirectional=cfg['bidirectional'],
).to(device)

# Loss y Optimizer
criterion = nn.BCEWithLogitsLoss()  # Combina sigmoid + BCE (mas estable numericamente)
optimizer = torch.optim.Adam(model.parameters(), lr=cfg['learning_rate'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

print(f"Parametros totales: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training Loop
epochs = cfg['epochs']
patience = cfg['patience']
train_losses, val_losses = [], []
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(epochs):
    # --- Train ---
    model.train()
    train_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        output = model(batch_x).squeeze(-1)
        loss = criterion(output, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping
        optimizer.step()
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    
    # --- Validation ---
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            output = model(batch_x).squeeze(-1)
            val_loss += criterion(output, batch_y).item()
    
    val_loss /= len(val_loader)
    scheduler.step(val_loss)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), str(get_path('models') / 'lstm_best.pt'))
    else:
        patience_counter += 1
    
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}')
    
    if patience_counter >= patience:
        print(f'Early stopping en epoch {epoch+1}')
        break

print(f'\nMejor Val Loss: {best_val_loss:.4f}')

In [ ]:
# Curvas de entrenamiento
fig = plot_training_history(train_losses, val_losses)
plt.show()

In [ ]:
# Evaluacion en test
model.load_state_dict(torch.load(str(get_path('models') / 'lstm_best.pt'), weights_only=True))
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        output = torch.sigmoid(model(batch_x).squeeze(-1))
        all_preds.extend(output.cpu().numpy())
        all_targets.extend(batch_y.numpy())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)

# Metricas
y_pred_binary = (all_preds > 0.5).astype(int)
print('Classification Report:')
print(classification_report(all_targets, y_pred_binary, target_names=['Normal', 'Falla 24h']))
print(f'ROC AUC: {roc_auc_score(all_targets, all_preds):.4f}')